# Credit Default Prediction using ANN

This notebook trains an **Artificial Neural Network (Multi-Layer Perceptron)** to predict credit risk tiers for Indian bank customers.

## Project Overview

- **Task**: 4-class classification — predict which credit risk tier a loan applicant falls into
- **Data Sources**: Internal bank records (PROSPECTID-linked) merged with external CIBIL bureau data
- **Target Classes**:
  - **P1** — Best (lowest risk, most creditworthy)
  - **P2** — Good
  - **P3** — Moderate risk
  - **P4** — High Risk (most likely to default)
- **Architecture**: 4-layer MLP with Batch Normalization, ReLU activations, and Dropout regularization
- **Framework**: PyTorch

### Workflow
1. Upload the two source Excel files
2. Merge, clean, and preprocess the data
3. Define and train the ANN
4. Evaluate on held-out test data
5. Visualize confusion matrix and metric bar chart

In [ ]:
!pip install torch pandas numpy scikit-learn matplotlib seaborn openpyxl

## Step 1: Upload Data Files

Run the cell below and upload **both** Excel files when prompted:

- `Internal_Bank_Dataset.xlsx` — internal bank records (applicant demographics, loan history, account behaviour)
- `External_Cibil_Dataset.xlsx` — CIBIL bureau data (credit scores, repayment history, outstanding balances)

> The files will be uploaded to the current Colab working directory (`/content/`) and merged on `PROSPECTID`.

In [ ]:
from google.colab import files

print("Please upload Internal_Bank_Dataset.xlsx and External_Cibil_Dataset.xlsx")
uploaded = files.upload()

print("\nFiles uploaded successfully:")
for filename in uploaded.keys():
    print(f"  - {filename}")

required = {'Internal_Bank_Dataset.xlsx', 'External_Cibil_Dataset.xlsx'}
missing = required - set(uploaded.keys())
if missing:
    print(f"\nWARNING: The following required files were not uploaded: {missing}")
    print("Re-run this cell and upload both files.")
else:
    print("\nBoth required files are present. Proceed to Step 2.")

## Step 2: Data Preparation

This step:
- Loads both Excel files
- Replaces sentinel value `-99999` with `NaN`
- Drops rows with missing `Approved_Flag` (target)
- Merges internal and external data on `PROSPECTID`
- Drops columns with more than 30% missing values
- Imputes remaining numeric NaNs with column medians; categorical NaNs with column mode
- Label-encodes the target (`Approved_Flag`: P1→0, P2→1, P3→2, P4→3)
- One-hot encodes remaining categorical features
- Splits into 80% train / 20% test (stratified)
- Applies `StandardScaler` (fit on train, transform both)
- Saves processed CSVs to `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

print("Starting data preprocessing...")

internal_df = pd.read_excel('Internal_Bank_Dataset.xlsx')
external_df = pd.read_excel('External_Cibil_Dataset.xlsx')

print(f"Internal dataset shape: {internal_df.shape}")
print(f"External dataset shape: {external_df.shape}")

def handle_missing(df):
    df = df.replace(-99999, np.nan)
    return df

internal_df = handle_missing(internal_df)
external_df = handle_missing(external_df)

external_df = external_df.dropna(subset=['Approved_Flag'])
print(f"External rows after dropping missing target: {len(external_df)}")

df = pd.merge(internal_df, external_df, on='PROSPECTID', how='inner')
print(f"Merged shape: {df.shape}")

threshold = 0.3 * len(df)
df = df.dropna(thresh=threshold, axis=1)
print(f"Shape after dropping sparse columns: {df.shape}")

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Merged shape after cleaning: {df.shape}")

target_encoder = LabelEncoder()
df['Approved_Flag'] = target_encoder.fit_transform(df['Approved_Flag'])
print(f"Target classes: {list(target_encoder.classes_)} -> {list(range(len(target_encoder.classes_)))}")

df = pd.get_dummies(df, drop_first=True)
print(f"Shape after one-hot encoding: {df.shape}")

drop_cols = ['Approved_Flag']
if 'PROSPECTID' in df.columns:
    drop_cols.append('PROSPECTID')

X = df.drop(drop_cols, axis=1)
y = df['Approved_Flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

os.makedirs('data/processed', exist_ok=True)
X_train_final.to_csv('data/processed/X_train.csv', index=False)
X_test_final.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print("\nPreprocessing complete.")
print(f"Final Train Shape: {X_train_final.shape}")
print(f"Final Test Shape:  {X_test_final.shape}")
print(f"Class distribution in train set:\n{y_train.value_counts().sort_index()}")

## Step 3: Build and Train ANN Model

### Architecture

```
Input (n_features)
  └── Linear(128) -> BatchNorm -> ReLU -> Dropout(0.3)
       └── Linear(64)  -> BatchNorm -> ReLU -> Dropout(0.2)
            └── Linear(32)  -> BatchNorm -> ReLU -> Dropout(0.2)
                 └── Linear(4)   [logits for P1, P2, P3, P4]
```

### Training Details
- **Loss**: CrossEntropyLoss with inverse-frequency class weights (handles class imbalance)
- **Optimizer**: Adam (lr=0.001, weight_decay=1e-5)
- **Batch size**: 64
- **Epochs**: 20

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import os

print("PyTorch Version:", torch.__version__)

X_train_df = pd.read_csv('data/processed/X_train.csv')
X_test_df  = pd.read_csv('data/processed/X_test.csv')
y_train_df = pd.read_csv('data/processed/y_train.csv')
y_test_df  = pd.read_csv('data/processed/y_test.csv')

X_train_t = torch.tensor(X_train_df.values, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_df.values,  dtype=torch.float32)
y_train_t = torch.tensor(y_train_df.values.ravel(), dtype=torch.long)
y_test_t  = torch.tensor(y_test_df.values.ravel(),  dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)


class CreditRiskANN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(CreditRiskANN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)


input_dimension = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))
print(f"Initializing ANN with input_dim={input_dimension} and num_classes={num_classes}")

model = CreditRiskANN(input_dimension, num_classes)

class_counts = torch.bincount(y_train_t)
total_samples = len(y_train_t)
class_weights = total_samples / (num_classes * class_counts.float())
print("Class Weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

num_epochs = 20
train_losses = []
print("\nStarting Training...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("\nEvaluating Model...")
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall    = recall_score(all_labels, all_preds, average='weighted')
f1        = f1_score(all_labels, all_preds, average='weighted')

n_classes = len(np.unique(all_labels))
if n_classes > 2:
    roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
else:
    roc_auc = roc_auc_score(all_labels, [p[1] for p in all_probs])

print(f"\n--- TEST SET METRICS ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

os.makedirs('saved_models', exist_ok=True)
torch.save(model.state_dict(), 'saved_models/ann_model.pth')
print("\nModel saved to saved_models/ann_model.pth")

## Step 4: Visualize Results

Two plots are generated inline:

1. **Confusion Matrix** — heatmap showing predicted vs actual risk tiers across all four classes (P1–P4)
2. **Metric Bar Chart** — bar chart comparing Accuracy, Precision, Recall, F1-Score, and ROC-AUC on the test set

Both plots are also saved to the `visualizations/` directory.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

os.makedirs('visualizations', exist_ok=True)

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.set(font_scale=1.2)
class_labels = ['P1 (Best)', 'P2 (Good)', 'P3 (Moderate)', 'P4 (High Risk)']
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Neural Network Confusion Matrix', fontsize=16, pad=20)
plt.ylabel('Actual True Risk Level', fontsize=14)
plt.xlabel('Predicted Risk Level', fontsize=14)
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to visualizations/confusion_matrix.png")

metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metric_values = [accuracy, precision, recall, f1, roc_auc]
bar_colors    = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

plt.figure(figsize=(10, 6))
bars = plt.bar(metric_names, metric_values, color=bar_colors, width=0.5, edgecolor='black')

for bar, value in zip(bars, metric_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{value:.4f}',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

plt.ylim(0, 1.15)
plt.title('ANN Model — Test Set Performance Metrics', fontsize=16, pad=15)
plt.ylabel('Score', fontsize=13)
plt.xlabel('Metric', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('visualizations/metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Metric bar chart saved to visualizations/metric_bars.png")